# 01 Quick Start

Create a compact playground with the main nbplay widgets. Run the cell, then open the tabs and interact with each surface.


In [ ]:
import ipywidgets as widgets

from nbplay import (
    EffectPlugin,
    KeyboardWidget,
    MidiKeyboardWidget,
    PadWidget,
    SamplerWidget,
    SequencerWidget,
    Session,
    SettingsWidget,
    SynthWidget,
)

# A session gives its transport, timeline and mixer one browser clock and audio bus.
session = Session(bpm=120.0)
settings = SettingsWidget()
transport = session.transport

synth = SynthWidget(oscillator_type="saw", frequency=440.0, amplitude=0.35)
sampler = SamplerWidget(sample_name="Drop audio here", pad_count=8)
seq = SequencerWidget(length=8, num_voices=2, bpm=120.0)
for i, note in enumerate([60, 64, 67, 72, 67, 64, 62, 59]):
    seq.set_step(i, note=note, velocity=96, active=i % 2 == 0, voice=0)

keyboard = KeyboardWidget(upper_octave=4, lower_octave=3, velocity=96)
midi_keyboard = MidiKeyboardWidget()
pads = PadWidget(rows=2, cols=4, velocity=110)

mixer = session.mixer
channel = mixer.add_channel("Preview")
mixer.add_channel_effect(channel, EffectPlugin("compressor", threshold=-18, ratio=4))
mixer.add_master_effect(EffectPlugin("limiter", threshold=-1))

timeline = session.timeline
timeline.config.update(length=64, count_in_bars=1, auto_extend_recording=True, recording_extend_bars=16)
timeline.add_track("Audio", channel_index=channel, armed=True, monitor=True)
timeline.add_clip("Drop or record here", track_index=0, start=0, duration=4, source="placeholder")

tabs = widgets.Tab([
    widgets.VBox([settings, transport]),
    widgets.VBox([synth, seq]),
    widgets.VBox([sampler, pads]),
    widgets.VBox([keyboard, midi_keyboard]),
    widgets.VBox([timeline, mixer]),
])
for index, title in enumerate(["Setup", "Synth + Seq", "Sampler + Pads", "Keys", "Timeline + Mixer"]):
    tabs.set_title(index, title)

tabs


## Themes

Every widget has a `theme` config field. Empty follows the Jupyter theme; `light` and `dark` pin the default look; `studio`, `flat`, `cube`, `one`, `grim`, `pro` and `fruit` are complete sets. `Session(theme=...)` or `session.theme` sets all of a session's widgets at once; here a dropdown drives every widget on the page, including the standalone ones.

In [ ]:
from nbplay import THEMES

themed = [settings, transport, synth, sampler, seq, keyboard, midi_keyboard, pads, mixer, timeline]
theme = widgets.Dropdown(options=THEMES, value="", description="Theme")


def apply_theme(change):
    for w in themed:
        w.config.theme = change["new"]


theme.observe(apply_theme, names="value")
theme